<a href="https://colab.research.google.com/github/jishnujs1990/Study/blob/master/Paraphraser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#AI-powered paraphrasing tool
**Develop an AI-powered paraphrasing tool as a Python application or module. The tool
should:**

● Take a block of input text

● Use deep learning (preferably transformer models like T5, BERT, GPT, or similar
via Hugging Face Transformers)

● Generate a paraphrased version preserving meaning, improving clarity, and
ensuring originality.

● Include built-in checks for grammar, spelling, and fluency of output

**Recommended technologies:**

● Python, with Hugging Face Transformers and relevant NLP packages (spaCy,
NLTK, etc.)

● TensorFlow or PyTorch for any model fine-tuning

● Use only console or script-based input/output (no GUI or web interface)

● Optional: Integrate evaluation metrics like BLEU, ROUGE, or semantic similarity
scores

**Expected Output:**

● Well-commented Python scripts/notebooks

● Sample results demonstrating paraphrased outputs

● Evaluation report showing tool accuracy and originality

In [1]:
!pip install -q torch transformers sentencepiece language-tool-python sentence-transformers rouge-score nltk

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 1.4 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import language_tool_python
from sentence_transformers import SentenceTransformer, util
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer


class AIParaphraser:
    """
    An AI-powered Paraphrasing Tool utilizing Hugging Face Transformers,
    automated grammar checks, and NLP evaluation metrics.
    """
    def __init__(self, model_name: str = "Vamsi/T5_Paraphrase_Paws"):
        print("Initializing Paraphraser Model...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        # Load T5 Paraphrase Model and Tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(self.device)

        # Load Grammar Checker
        print("Loading Grammar Checker...")
        self.grammar_tool = language_tool_python.LanguageTool('en-US')

        # Load Sentence Transformer for Semantic Similarity Evaluation
        print("Loading Semantic Evaluation Model...")
        self.similarity_model = SentenceTransformer('all-MiniLM-L6-v2')

        # ROUGE Scorer Setup
        self.rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

    def paraphrase(self, text: str, num_return_sequences: int = 1, max_length: int = 256) -> list:
        """
        Generates paraphrased versions of the input text using T5.
        """
        # Prepend task prefix required by T5 fine-tuned models
        text_prefix = "paraphrase: " + text + " </s>"

        inputs = self.tokenizer(
            text_prefix,
            padding="longest",
            max_length=max_length,
            truncation=True,
            return_tensors="pt"
        ).to(self.device)

        # Beam search decoding for high-quality sequence generation
        outputs = self.model.generate(
            **inputs,
            num_beams=5,
            num_return_sequences=num_return_sequences,
            max_length=max_length,
            no_repeat_ngram_size=2,
            early_stopping=True
        )

        results = [
            self.tokenizer.decode(out, skip_special_tokens=True, clean_up_tokenization_spaces=True)
            for out in outputs
        ]
        return results

    def check_and_correct_grammar(self, text: str) -> tuple:
        """
        Checks input text for grammatical errors and returns (corrected_text, match_count).
        """
        matches = self.grammar_tool.check(text)
        corrected_text = language_tool_python.utils.correct(text, matches)
        return corrected_text, len(matches)

    def evaluate(self, original_text: str, paraphrased_text: str) -> dict:
        """
        Calculates semantic similarity, BLEU score, and ROUGE score between original and generated text.
        """
        # 1. Semantic Similarity (Sentence Transformers Cosine Similarity)
        emb_orig = self.similarity_model.encode(original_text, convert_to_tensor=True)
        emb_para = self.similarity_model.encode(paraphrased_text, convert_to_tensor=True)
        semantic_sim = float(util.cos_sim(emb_orig, emb_para)[0][0])

        # 2. BLEU Score
        ref_tokens = [original_text.lower().split()]
        hyp_tokens = paraphrased_text.lower().split()
        smooth = SmoothingFunction().method1
        bleu_score = sentence_bleu(ref_tokens, hyp_tokens, smoothing_function=smooth)

        # 3. ROUGE Scores
        rouge_scores = self.rouge.score(original_text, paraphrased_text)

        return {
            "semantic_similarity": round(semantic_sim, 4),
            "bleu_score": round(bleu_score, 4),
            "rouge1_f1": round(rouge_scores['rouge1'].fmeasure, 4),
            "rougeL_f1": round(rouge_scores['rougeL'].fmeasure, 4)
        }

    def process_pipeline(self, text: str) -> dict:
        """
        Runs full end-to-end pipeline: Paraphrase -> Grammar Check -> Evaluation.
        """
        # Step 1: Generate Paraphrase
        raw_paraphrases = self.paraphrase(text, num_return_sequences=1)
        raw_paraphrase = raw_paraphrases[0]

        # Step 2: Grammar Correction & Fluency Check
        corrected_paraphrase, error_count = self.check_and_correct_grammar(raw_paraphrase)

        # Step 3: Evaluate Accuracy and Originality
        metrics = self.evaluate(text, corrected_paraphrase)

        return {
            "original": text,
            "raw_paraphrase": raw_paraphrase,
            "final_paraphrase": corrected_paraphrase,
            "grammar_corrections_made": error_count,
            "metrics": metrics
        }


# =====================================================================
# Console Input/Output Execution
# =====================================================================
if __name__ == "__main__":
    paraphraser = AIParaphraser()

    print("\n" + "="*60)
    print("        AI-POWERED PARAPHRASING TOOL (CONSOLE INTERFACE)")
    print("="*60)

    sample_inputs = [
        "Deep learning models require large amounts of data to achieve high performance on complex tasks.",
        "The quick brown fox jumps over the lazy dog in the middle of the quiet forest.",
        "Artificial intelligence is rapidly transforming industries by automating repetitive tasks and providing data-driven insights."
    ]

    for idx, sample in enumerate(sample_inputs, 1):
        print(f"\n--- Test Case {idx} ---")
        result = paraphraser.process_pipeline(sample)

        print(f"\n[Original Text]:\n  {result['original']}")
        print(f"\n[Paraphrased Output]:\n  {result['final_paraphrase']}")
        print(f"\n[Grammar Issues Detected & Corrected]: {result['grammar_corrections_made']}")
        print("\n[Evaluation Metrics]:")
        for metric_name, val in result['metrics'].items():
            print(f"  - {metric_name.replace('_', ' ').title()}: {val}")
        print("-" * 60)

Initializing Paraphraser Model...


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


        AI-POWERED PARAPHRASING TOOL (CONSOLE INTERFACE)

--- Test Case 1 ---

[Original Text]:
  Deep learning models require large amounts of data to achieve high performance on complex tasks.

[Paraphrased Output]:
  Deep learning models require large amounts of data to achieve high performance on complex tasks.

[Grammar Issues Detected & Corrected]: 0

[Evaluation Metrics]:
  - Semantic Similarity: 1.0
  - Bleu Score: 1.0
  - Rouge1 F1: 1.0
  - Rougel F1: 1.0
------------------------------------------------------------

--- Test Case 2 ---

[Original Text]:
  The quick brown fox jumps over the lazy dog in the middle of the quiet forest.

[Paraphrased Output]:
  The quick brown fox jumps over the lazy dog in the middle of the quiet forest.

[Grammar Issues Detected & Corrected]: 0

[Evaluation Metrics]:
  - Semantic Similarity: 1.0
  - Bleu Score: 1.0
  - Rouge1 F1: 1.0
  - Rougel F1: 1.0
------------------------------------------------------------

--- Test Case 3 ---

[Original 

In [3]:
if __name__ == "__main__":
    paraphraser = AIParaphraser()

    print("\n" + "="*60)
    print("        AI-POWERED PARAPHRASING TOOL (CONSOLE INTERFACE)")
    print("="*60)

    sample_inputs = [
        "India is our country.",
        "India is our Country.",
        "Apple and Banana is fruits."
    ]

    for idx, sample in enumerate(sample_inputs, 1):
        print(f"\n--- Test Case {idx} ---")
        result = paraphraser.process_pipeline(sample)

        print(f"\n[Original Text]:\n  {result['original']}")
        print(f"\n[Paraphrased Output]:\n  {result['final_paraphrase']}")
        print(f"\n[Grammar Issues Detected & Corrected]: {result['grammar_corrections_made']}")
        print("\n[Evaluation Metrics]:")
        for metric_name, val in result['metrics'].items():
            print(f"  - {metric_name.replace('_', ' ').title()}: {val}")
        print("-" * 60)

Initializing Paraphraser Model...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Loading Grammar Checker...
Loading Semantic Evaluation Model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


        AI-POWERED PARAPHRASING TOOL (CONSOLE INTERFACE)

--- Test Case 1 ---

[Original Text]:
  India is our country.

[Paraphrased Output]:
  India is our country.

[Grammar Issues Detected & Corrected]: 0

[Evaluation Metrics]:
  - Semantic Similarity: 1.0
  - Bleu Score: 1.0
  - Rouge1 F1: 1.0
  - Rougel F1: 1.0
------------------------------------------------------------

--- Test Case 2 ---

[Original Text]:
  India is our Country.

[Paraphrased Output]:
  India is our country.

[Grammar Issues Detected & Corrected]: 0

[Evaluation Metrics]:
  - Semantic Similarity: 1.0
  - Bleu Score: 1.0
  - Rouge1 F1: 1.0
  - Rougel F1: 1.0
------------------------------------------------------------

--- Test Case 3 ---

[Original Text]:
  Apple and Banana is fruits.

[Paraphrased Output]:
  Apple and banana are fruits.

[Grammar Issues Detected & Corrected]: 0

[Evaluation Metrics]:
  - Semantic Similarity: 0.9892
  - Bleu Score: 0.2857
  - Rouge1 F1: 0.8
  - Rougel F1: 0.8
---------------

In [6]:
user_input = input("Enter a sentence to paraphrase: ")
result = paraphraser.process_pipeline(user_input)

print("\n--- Result ---")
print("Original:  ", result['original'])
print("Paraphrase:", result['final_paraphrase'])
print("Metrics:   ", result['metrics'])

Enter a sentence to paraphrase: What are your name?

--- Result ---
Original:   What are your name?
Paraphrase: What is your name?
Metrics:    {'semantic_similarity': 0.9791, 'bleu_score': 0.188, 'rouge1_f1': 0.75, 'rougeL_f1': 0.75}


In [7]:
user_input = input("Enter a sentence to paraphrase: ")
result = paraphraser.process_pipeline(user_input)

print("\n--- Result ---")
print("Original:  ", result['original'])
print("Paraphrase:", result['final_paraphrase'])
print("Metrics:   ", result['metrics'])

Enter a sentence to paraphrase: What is your Name?

--- Result ---
Original:   What is your Name?
Paraphrase: What is your name?
Metrics:    {'semantic_similarity': 1.0, 'bleu_score': 1.0, 'rouge1_f1': 1.0, 'rougeL_f1': 1.0}
